# Notebook 16 – Pre-trained Model Integration

## Topic 1: Project Plan and Setup

**Concept**
We build a small **Customer Review Analyzer** that combines two different pre-trained models:

```
Review text
   ↓
Model 1: Sentiment Classifier  → POSITIVE / NEGATIVE
   ↓
Model 2: Named Entity Recognition → people, organizations, locations
   ↓
Combined Result
```

The code is split into small modules inside a `review_app/` package: `config`, `loaders`, `sentiment`, `ner`, `pipeline`.

**Why it's needed**
Real AI systems rarely use one model. A modular design lets us swap, test, or fix one model without touching the rest.

In [ ]:
import os, sys

# Install once if needed (uncomment):
# !pip install transformers torch

os.makedirs("review_app", exist_ok=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Package folder ready:", os.path.abspath("review_app"))

**What this example tells us**
We prepared a package folder. Each later topic writes one small module into it.

**AI/ML Example**
A production NLP service usually has separate files for config, model loading, and inference.

**Business Example**
A support team wants every customer review tagged with mood and mentioned companies or places.

**AI/ML Engineer Use Case**
Setting up a clean project layout first makes the pipeline easy to test and deploy later.

## Topic 2: Configuration Module

**Concept**
`config.py` stores the model names in one place.

**Why it's needed**
Changing a model should mean editing one line, not hunting through the code.

In [ ]:
%%writefile review_app/config.py
SENTIMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
NER_MODEL = "dslim/bert-base-NER"

**What this example tells us**
Model names now live in `config.py`. Nothing else in the app hard-codes them.

**AI/ML Example**
Experiment configs often hold model names, thresholds, and batch sizes so runs are reproducible.

**Business Example**
If the company switches to a better sentiment model, only one line changes.

**AI/ML Engineer Use Case**
Central config makes A/B testing two model versions simple.

## Topic 3: Loader Module

**Concept**
`loaders.py` wraps `transformers.pipeline` in one function.

**Why it's needed**
All models load the same way, so error handling and device choice can be added in one place later.

In [ ]:
%%writefile review_app/loaders.py
from transformers import pipeline


def load_pipeline(task, model_name, **kwargs):
    """Load any Hugging Face pipeline with a single, shared function."""
    return pipeline(task, model=model_name, **kwargs)

**What this example tells us**
Both models will be loaded through the same function, so loading logic is not repeated.

**AI/ML Example**
Teams often add GPU selection or caching inside one loader function.

**Business Example**
One loader means fewer bugs when the company adds a third model later.

**AI/ML Engineer Use Case**
A single loading entry point is the place to add logging, timing, and device settings.

## Topic 4: Package Marker

**Concept**
An empty `__init__.py` tells Python that `review_app` is a package, so we can write `from review_app.pipeline import ...`.

**Why it's needed**
Without it, imports between our modules would not work cleanly.

In [ ]:
%%writefile review_app/__init__.py
# review_app package

**What this example tells us**
The folder is now importable as a package.

**AI/ML Example**
Every reusable ML library (scikit-learn, Transformers) is built as a package.

**Business Example**
Other teams can `import review_app` without copying files.

**AI/ML Engineer Use Case**
Packaging lets you pip-install your pipeline inside Docker images.

## Topic 5: Model 1 – Sentiment Module

**Concept**
`SentimentAnalyzer` wraps a pre-trained text classifier. Input: text. Output: a label (POSITIVE/NEGATIVE) and a confidence score.

**Why it's needed**
Wrapping the model in a class hides library details. The rest of the app only calls `.predict(text)`.

In [ ]:
%%writefile review_app/sentiment.py
from .config import SENTIMENT_MODEL
from .loaders import load_pipeline


class SentimentAnalyzer:
    def __init__(self, model_name=SENTIMENT_MODEL):
        self.model = load_pipeline("sentiment-analysis", model_name)

    def predict(self, text):
        result = self.model(text)[0]
        return {"label": result["label"], "score": round(result["score"], 3)}

**What this example tells us**
Model 1 is now a clean, reusable component with a simple `predict` method.

**AI/ML Example**
Classification heads like this power spam filters and review rating systems.

**Business Example**
Flagging NEGATIVE reviews lets support teams respond quickly.

**AI/ML Engineer Use Case**
A stable `predict` interface lets you replace the model without changing downstream code.

## Topic 6: Model 2 – NER Module

**Concept**
`EntityExtractor` wraps a pre-trained Named Entity Recognition model. It finds names of people (PER), organizations (ORG), locations (LOC), and miscellaneous items (MISC).

**Why it's needed**
Sentiment says *how* a customer feels. NER says *who or what* they are talking about.

In [ ]:
%%writefile review_app/ner.py
from .config import NER_MODEL
from .loaders import load_pipeline


class EntityExtractor:
    def __init__(self, model_name=NER_MODEL):
        self.model = load_pipeline(
            "ner", model_name, aggregation_strategy="simple"
        )

    def predict(self, text):
        entities = self.model(text)
        return [
            {"text": e["word"], "type": e["entity_group"], "score": round(float(e["score"]), 3)}
            for e in entities
        ]

**What this example tells us**
Model 2 has the same style of interface as Model 1, which makes combining them easy.

**AI/ML Example**
NER is used to extract names and places from news, legal, and medical documents.

**Business Example**
Finding which store branch or product a customer complained about.

**AI/ML Engineer Use Case**
Grouping sub-word tokens into whole entities (`aggregation_strategy="simple"`) gives clean output for downstream systems.

## Topic 7: Pipeline Module (Combining Both Models)

**Concept**
`ReviewAnalyzer` is the orchestrator. It owns both models, sends the same text through each, and merges the results into one dictionary.

**Why it's needed**
Integration logic lives in one place. Individual models stay independent and replaceable.

In [ ]:
%%writefile review_app/pipeline.py
from .sentiment import SentimentAnalyzer
from .ner import EntityExtractor


class ReviewAnalyzer:
    def __init__(self):
        self.sentiment = SentimentAnalyzer()
        self.entities = EntityExtractor()

    def analyze(self, text):
        return {
            "text": text,
            "sentiment": self.sentiment.predict(text),
            "entities": self.entities.predict(text),
        }

    def analyze_many(self, texts):
        return [self.analyze(t) for t in texts]

**What this example tells us**
Two separate models are combined behind a single `analyze` method.

**AI/ML Example**
Multi-model pipelines (ASR → NER, OCR → classifier) follow this same orchestrator pattern.

**Business Example**
One call returns mood and mentioned entities for each review.

**AI/ML Engineer Use Case**
The orchestrator is the natural place to add batching, caching, and error handling.

## Topic 8: Running the Application

**Concept**
We import the package, create `ReviewAnalyzer` (this downloads both models the first time), and analyze sample reviews.

**Why it's needed**
This proves the modules work together end to end.

In [ ]:
from review_app.pipeline import ReviewAnalyzer

reviews = [
    "The delivery from Amazon to Chennai was fast and the staff was very helpful.",
    "Very disappointed with Samsung support in Bangalore, they ignored my complaint.",
    "Tim Cook announced a great new product at Apple Park.",
]

app = ReviewAnalyzer()
results = app.analyze_many(reviews)

for r in results:
    print("Text      :", r["text"])
    print("Sentiment :", r["sentiment"])
    print("Entities  :", [(e["text"], e["type"]) for e in r["entities"]])
    print("-" * 60)

**What this example tells us**
Each review gets a sentiment label plus the entities it mentions, from two different models, in one result.
(Run the cell to see your own output; exact scores depend on the downloaded model weights.)

**AI/ML Example**
Same pattern as a production NLP API returning several model outputs per request.

**Business Example**
A dashboard can show "negative reviews mentioning Samsung in Bangalore" from this output.

**AI/ML Engineer Use Case**
Testing the full flow on a few sample inputs is the first step before adding an API layer.